# Genetic Algorithm — binary example from start to finish

This notebook shows the program working on the binary dataset (`binary_data_1.xlsx`, classes 0 and 1).

1. **Step by step:** runs one generation by hand, calling each function in the order of the algorithm
2. **Full program:** uses `train()` to train the model and `predict()` to classify new data

## 0. Imports

In [1]:
import sys
sys.path.insert(0, "../../src")  # folder that holds the genetic_classifier package

import numpy as np

from genetic_classifier.reading import read_excel
from genetic_classifier.creation import create_chromosomes
from genetic_classifier.fitness import binary_fitness, fitness_proportions
from genetic_classifier.selection import select_parents_roulette
from genetic_classifier.crossover import one_point_crossover
from genetic_classifier.mutation import mutate
from genetic_classifier.update import update_population
from genetic_classifier import train, predict

np.random.seed(0)  # seed: running again gives the same random draws

## 1. Reading the data

`read_excel` ignores the first column (lesson names), uses the last one as the class and the rest as features.

In [2]:
X, y = read_excel("binary_data_1.xlsx")

print("X (n_samples, n_features):", X.shape)
print("y (n_samples,):", y.shape)
print("class 0:", sum(y == 0), "| class 1:", sum(y == 1))
print("\nfirst sample:", X[0], "-> class", y[0])

X (n_samples, n_features): (539, 18)
y (n_samples,): (539,)
class 0: 261 | class 1: 278

first sample: [1 0 0 1 1 0 0 1 0 1 1 0 0 1 1 0 1 0] -> class 1


## 2. Splitting train and test

80% of the data for training and 20% for testing. The test shows whether the model gets right data it has **never seen**.

`np.random.permutation(n)` returns the numbers 0 to n-1 shuffled, and we use that as the order of the samples.

In [3]:
order = np.random.permutation(len(y))
cut = int(0.8 * len(y))

X_train, y_train = X[order[:cut]], y[order[:cut]]
X_test, y_test = X[order[cut:]], y[order[cut:]]

print("train:", X_train.shape, "| test:", X_test.shape)

train: (431, 18) | test: (108, 18)


---
# Part 1 — One generation step by step

### Step 1: `create_chromosomes`
Each chromosome has 1 bias gene and 1 weight for each feature.

In [4]:
n_genes = X_train.shape[1] + 1  # features + bias
population = create_chromosomes(n_chromosomes=6, n_genes=n_genes)

print("population:", population.shape)
print("first chromosome:", population[0].round(2))

population: (6, 19)
first chromosome: [ 0.24  0.73 -0.68  0.98 -0.83 -0.81  0.27  0.9   0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]


### Step 2: `binary_fitness` and `fitness_proportions`
Fitness = recall of class 1 × recall of class 0, so it goes from 0 to 1.
Then the fitnesses become the chances of each chromosome in the roulette.

In [5]:
fitnesses = binary_fitness(population, X_train, y_train)
proportions = fitness_proportions(fitnesses)

for i in range(len(population)):
    print(f"chromosome {i}: fitness = {fitnesses[i]:.4f} | roulette chance = {proportions[i]:.2%}")

chromosome 0: fitness = 0.0827 | roulette chance = 10.69%
chromosome 1: fitness = 0.2062 | roulette chance = 26.67%
chromosome 2: fitness = 0.2204 | roulette chance = 28.50%
chromosome 3: fitness = 0.0876 | roulette chance = 11.33%
chromosome 4: fitness = 0.0266 | roulette chance = 3.44%
chromosome 5: fitness = 0.1498 | roulette chance = 19.37%


### Step 3: `select_parents_roulette`
The higher the fitness, the higher the chance of being picked as a parent.

In [6]:
parent1, parent2 = select_parents_roulette(population, proportions)

print("parent1:", parent1.round(2))
print("parent2:", parent2.round(2))

parent1: [-0.74  0.42  0.42  0.22  0.34 -0.1   0.39 -0.24 -0.8   0.92 -0.86 -0.96
  0.96 -0.7  -0.48 -0.57  0.16  0.95 -0.45]
parent2: [ 0.24  0.73 -0.68  0.98 -0.83 -0.81  0.27  0.9   0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]


### Step 4: `one_point_crossover`
The child takes the beginning of parent1 and, from a random cut point onwards, the end of parent2.

In [7]:
offspring = one_point_crossover(parent1, parent2, n_offspring=3)

for i, child in enumerate(offspring):
    print(f"child {i}:", child.round(2))

child 0: [-0.74  0.42  0.42  0.22  0.34 -0.1   0.39 -0.24  0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]
child 1: [-0.74  0.42  0.42  0.22  0.34 -0.1   0.39 -0.24 -0.8   0.92 -0.86  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]
child 2: [-0.74  0.42  0.42  0.22 -0.83 -0.81  0.27  0.9   0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]


### Step 5: `mutate`
Some genes of each child are replaced by a random value between -1 and 1.

In [8]:
mutated_offspring = mutate(offspring, n_mutated_genes=1)

for i in range(len(offspring)):
    changed = [g for g in range(n_genes) if offspring[i][g] != mutated_offspring[i][g]]
    print(f"child {i}: mutated gene(s) = {changed}")

child 0: mutated gene(s) = [1]
child 1: mutated gene(s) = [15]
child 2: mutated gene(s) = [15]


### Step 6: `update_population`
The worst chromosomes of the population leave and the best children take their place.

In [9]:
offspring_fitnesses = binary_fitness(mutated_offspring, X_train, y_train)
print("offspring fitness:", offspring_fitnesses.round(4))

new_population = update_population(population, fitnesses, mutated_offspring, offspring_fitnesses, n_replacements=2)
new_fitnesses = binary_fitness(new_population, X_train, y_train)

print("\nfitness before:", fitnesses.round(4), "| best:", round(fitnesses.max(), 4))
print("fitness after: ", new_fitnesses.round(4), "| best:", round(new_fitnesses.max(), 4))

offspring fitness: [0.4515 0.1933 0.2531]

fitness before: [0.0827 0.2062 0.2204 0.0876 0.0266 0.1498] | best: 0.2204
fitness after:  [0.4515 0.2062 0.2204 0.0876 0.2531 0.1498] | best: 0.4515


That was **one generation**. The algorithm repeats steps 2 to 6 until it reaches the target fitness or runs out of generations.

---
# Part 2 — Full program

### Training with `train`
Use `verbose=True` to see the best fitness of every generation.

In [10]:
np.random.seed(0)

model = train(
    X_train, y_train,
    n_chromosomes=6,
    n_mutated_genes=1,
    n_offspring=3,
    n_replacements=2,
    n_generations=500,
    target_fitness=0.90,
    verbose=False,
)

print("best fitness:", round(model["fitness"], 4))
print("stopped at generation:", model["generation"])
print("best chromosome:", model["chromosome"].round(2))

best fitness: 0.8216
stopped at generation: 500
best chromosome: [ 0.32 -0.92  0.84 -0.63  0.39  0.63 -0.06  0.47 -0.35 -0.89 -0.46  0.44
  0.01 -0.55  0.75  0.22  0.69 -0.26 -0.41]


### Testing with `predict`
Classifies the test samples and compares them with the true class.

In [11]:
y_pred = predict(model, X_test)

hits = sum(y_pred == y_test)
print(f"test accuracy: {hits}/{len(y_test)} = {hits / len(y_test):.2%}")

# confusion matrix: how many of each true class were predicted as 0 or 1
print("\n            predicted 0  predicted 1")
for true in [0, 1]:
    p0 = sum((y_test == true) & (y_pred == 0))
    p1 = sum((y_test == true) & (y_pred == 1))
    print(f"true {true}       {p0:>8}     {p1:>8}")

test accuracy: 94/108 = 87.04%

            predicted 0  predicted 1
true 0             48            3
true 1             11           46


### Classifying a single sample

In [12]:
sample = X_test[0]
print("sample:", sample)
print("predicted class:", predict(model, sample)[0], "| true class:", y_test[0])

sample: [1 0 0 1 1 0 0 1 0 1 1 0 0 1 1 0 1 0]
predicted class: 1 | true class: 1
